# Tesla — Financial & Product Growth Analysis (2012–2020)

This Jupyter notebook is a self-contained analysis of Tesla's financials and vehicle deliveries for 2012–2020.

Files saved alongside this notebook:

- tesla_financials_2012_2020.csv
- tesla_deliveries_2012_2020.csv

All values are in **millions USD** unless noted otherwise. Vehicle counts are absolute units.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Load data (already saved alongside notebook)
financials = pd.read_csv('/mnt/data/tesla_project_outputs/tesla_financials_2012_2020.csv')
deliveries = pd.read_csv('/mnt/data/tesla_project_outputs/tesla_deliveries_2012_2020.csv')
financials['Year'] = financials['Year'].astype(int)
financials = financials.sort_values('Year').reset_index(drop=True)
deliveries['Year'] = deliveries['Year'].astype(int)
deliveries = deliveries.sort_values('Year').reset_index(drop=True)

# display heads
financials.head()


In [ ]:
# Derived metrics
financials['YoY_Revenue_Growth_pct'] = financials['Total_Revenues'].pct_change() * 100
financials['Gross_Margin_pct'] = financials['Gross_Profit'] / financials['Total_Revenues'] * 100
financials['Operating_Margin_pct'] = financials['Income_from_Operations'] / financials['Total_Revenues'] * 100
financials['Net_Margin_pct'] = financials['Net_Income'] / financials['Total_Revenues'] * 100

# FCF proxy: CFO - Delta PPE (proxy for capex)
financials['Delta_PPE'] = financials['PPE_net'].diff().fillna(0)
financials['FCF_proxy'] = financials['Net_Cash_from_Operations'] - financials['Delta_PPE']

financials[['Year','Total_Revenues','YoY_Revenue_Growth_pct','Gross_Margin_pct','Operating_Margin_pct','Net_Margin_pct','Net_Cash_from_Operations','FCF_proxy']]


In [ ]:
# Revenue trend (values in millions USD)
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(financials['Year'], financials['Total_Revenues'], marker='o')
ax.set_title('Tesla — Total Revenues (2012–2020)')
ax.set_xlabel('Year')
ax.set_ylabel('Revenue (millions USD)')
ax.grid(axis='y', linestyle='--', alpha=0.4)

# compute CAGR 2012-2020
rev_2012 = financials.loc[financials['Year']==2012, 'Total_Revenues'].values[0]
rev_2020 = financials.loc[financials['Year']==2020, 'Total_Revenues'].values[0]
years = 2020 - 2012
cagr = (rev_2020/rev_2012)**(1/years) - 1
ax.annotate(f'CAGR (2012-2020): {cagr*100:.1f}%', xy=(0.02, 0.95), xycoords='axes fraction', fontsize=10, bbox=dict(boxstyle='round', fc='w'))
plt.tight_layout()
plt.show()


In [ ]:
# Margin trends
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(financials['Year'], financials['Gross_Margin_pct'], marker='o', label='Gross Margin (%)')
ax.plot(financials['Year'], financials['Operating_Margin_pct'], marker='o', label='Operating Margin (%)')
ax.plot(financials['Year'], financials['Net_Margin_pct'], marker='o', label='Net Margin (%)')
ax.set_title('Tesla — Margin Trends (2012–2020)')
ax.set_xlabel('Year')
ax.set_ylabel('Margin (%)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# Deliveries vs Revenue
# approximate revenue per vehicle (Total_Revenues / Deliveries)
merged = pd.merge(financials[['Year','Total_Revenues']], deliveries[['Year','Deliveries']], on='Year', how='left')
merged['Revenue_per_Delivery'] = merged['Total_Revenues'] / merged['Deliveries']

fig, ax = plt.subplots(figsize=(10,5))
ax2 = ax.twinx()
ax.bar(merged['Year'], merged['Deliveries'], alpha=0.6)
ax.set_xlabel('Year')
ax.set_ylabel('Deliveries (units)')
ax2.plot(merged['Year'], merged['Total_Revenues'], marker='o')
ax2.set_ylabel('Total Revenues (millions USD)')
ax.set_title('Deliveries (bars) vs Total Revenues (line)')
plt.tight_layout()
plt.show()

merged[['Year','Deliveries','Total_Revenues','Revenue_per_Delivery']]


In [ ]:
# Cash flow key items (table)
financials[['Year','Net_Cash_from_Operations','Net_Cash_from_Investing','Net_Cash_from_Financing','Cash_End']]


In [ ]:
# Files produced alongside this notebook
print('Saved files:')
import os
for f in os.listdir('/mnt/data/tesla_project_outputs'):
    print('-', f)

print('\nCleaned financials CSV path: /mnt/data/tesla_project_outputs/tesla_financials_2012_2020.csv')
print('Deliveries CSV path: /mnt/data/tesla_project_outputs/tesla_deliveries_2012_2020.csv')
